# 🚁 AOD-4: YOLOv8n (nano)

**ВКР — визуальный модуль: обнаружение летающих объектов**

Датасет: **AOD-4** (22 516 img, 4 класса: Bird, Drone, Helicopter, Plane)  
Модель: **YOLOv8n** | epochs=80 | batch=16 | img=640

**Подключи датасет:** Add Data → `airborne-object-detection`

In [ ]:
!pip install -q ultralytics

In [ ]:
import os, json, yaml, shutil, random, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, cv2
from collections import Counter
from IPython.display import Image as IPImage, display
import torch
from ultralytics import YOLO
warnings.filterwarnings('ignore')

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

WORK = Path('/kaggle/working')
VIZ = WORK / 'visualizations'
RUNS = WORK / 'runs'
for d in [VIZ, RUNS]: d.mkdir(parents=True, exist_ok=True)

RUN_NAME = 'yolov8n_aod4'
IMG_SIZE = 640
print('✅ Готово')

---
## 1. Восстановление чекпоинтов (если есть)

In [ ]:
# Ищем чекпоинты от предыдущих запусков в /kaggle/input
restored = False
for d in sorted(Path('/kaggle/input').iterdir()):
    if d.name == 'airborne-object-detection':
        continue  # это датасет, не чекпоинт
    weights = list(d.rglob('last.pt')) + list(d.rglob('best.pt'))
    if weights:
        print(f'📦 Найден предыдущий запуск: {d.name}')
        for w in weights:
            rel = w.relative_to(d)
            dst = RUNS / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(w, dst)
            print(f'   ↻ скопирован {rel}')
        restored = True
        break

if not restored:
    print('🆕 Чекпоинтов нет — первый запуск')

---
## 2. Подготовка датасета

In [ ]:
SRC = Path('/kaggle/input/airborne-object-detection')
DS = WORK / 'dataset'

# Если уже подготовлен — пропускаем
if (DS / 'data.yaml').exists():
    print('✅ Датасет уже подготовлен')
else:
    assert (SRC / 'train' / 'images').exists(), f'Не найдено: {SRC}/train/images'
    print(f'📂 Источник: {SRC}')

    all_imgs = sorted((SRC/'train'/'images').glob('*.jpg'))
    all_imgs += sorted((SRC/'train'/'images').glob('*.png'))
    print(f'   Изображений: {len(all_imgs)}')

    random.shuffle(all_imgs)
    n = len(all_imgs)
    n_train = int(n * 0.80)
    n_val = int(n * 0.15)
    splits = {'train': all_imgs[:n_train],
              'valid': all_imgs[n_train:n_train+n_val],
              'test':  all_imgs[n_train+n_val:]}

    for sname, slist in splits.items():
        (DS/sname/'images').mkdir(parents=True, exist_ok=True)
        (DS/sname/'labels').mkdir(parents=True, exist_ok=True)
        for p in slist:
            shutil.copy2(p, DS/sname/'images'/p.name)
            lbl = SRC/'train'/'labels'/f'{p.stem}.txt'
            if lbl.exists(): shutil.copy2(lbl, DS/sname/'labels'/lbl.name)
        print(f'   {sname:5s}: {len(slist)}')

    # Читаем классы из оригинального yaml
    with open(SRC/'data.yaml') as f:
        orig = yaml.safe_load(f)
    names = orig.get('names', {0:'Bird',1:'Drone',2:'Helicopter',3:'Plane'})
    if isinstance(names, list): names = {i:n for i,n in enumerate(names)}
    nc = len(names)

    cfg = {'path':str(DS),'train':'train/images','val':'valid/images',
           'test':'test/images','names':names,'nc':nc}
    with open(DS/'data.yaml','w') as f: yaml.dump(cfg, f)
    print(f'\n✅ data.yaml создан ({nc} классов: {list(names.values())})')

YAML_PATH = DS / 'data.yaml'
with open(YAML_PATH) as f: ds_cfg = yaml.safe_load(f)
CLASS_NAMES = list(ds_cfg['names'].values()) if isinstance(ds_cfg['names'],dict) else ds_cfg['names']
NUM_CLASSES = len(CLASS_NAMES)
ds_root = DS
print(f'\n📋 Классы: {CLASS_NAMES}')

---
## 3. EDA

In [ ]:
# Подсчёт аннотаций
class_counts = Counter()
split_info = {}
for split in ['train','valid','test']:
    lbl_dir = ds_root/split/'labels'
    img_dir = ds_root/split/'images'
    if not img_dir.exists(): continue
    n_imgs = len(list(img_dir.iterdir()))
    for lf in lbl_dir.glob('*.txt'):
        for line in open(lf):
            parts = line.strip().split()
            if parts: class_counts[int(parts[0])] += 1
    split_info[split] = n_imgs
    print(f'   {split:5s}: {n_imgs} images')

print(f'\n📊 Классы:')
for i in sorted(class_counts): print(f'   [{i}] {CLASS_NAMES[i]:12s}: {class_counts[i]}')

# График
fig, ax = plt.subplots(figsize=(8,4))
names_plot = [CLASS_NAMES[i] for i in sorted(class_counts)]
vals = [class_counts[i] for i in sorted(class_counts)]
bars = ax.bar(names_plot, vals, color=['#2ecc71','#e74c3c','#3498db','#f39c12'][:len(names_plot)])
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2,v+50,str(v),ha='center',fontweight='bold')
ax.set_title('AOD-4: распределение классов'); ax.set_ylabel('Аннотаций'); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig(VIZ/'class_distribution.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# Примеры с bbox
img_dir = ds_root/'train'/'images'
lbl_dir = ds_root/'train'/'labels'
all_imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg','.png'}]
samples = random.sample(all_imgs, min(8, len(all_imgs)))

COLORS = [(0,200,0),(0,0,255),(255,165,0),(0,255,255)]
fig,axes = plt.subplots(2,4,figsize=(20,10)); axes=axes.flatten()
for ax,p in zip(axes,samples):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h,w = img.shape[:2]
    lf = lbl_dir/f'{p.stem}.txt'
    if lf.exists():
        for line in open(lf):
            parts = line.strip().split()
            if len(parts)>=5:
                ci=int(parts[0]); cx,cy,bw,bh=map(float,parts[1:5])
                x1,y1=int((cx-bw/2)*w),int((cy-bh/2)*h)
                x2,y2=int((cx+bw/2)*w),int((cy+bh/2)*h)
                c=COLORS[ci%len(COLORS)]
                cv2.rectangle(img,(x1,y1),(x2,y2),c,2)
                cv2.putText(img,CLASS_NAMES[ci],(x1,y1-5),cv2.FONT_HERSHEY_SIMPLEX,0.6,c,2)
    ax.imshow(img); ax.axis('off')
for j in range(len(samples),len(axes)): axes[j].axis('off')
plt.suptitle('AOD-4: примеры с разметкой',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig(VIZ/'sample_images.png',dpi=150,bbox_inches='tight'); plt.show()

---
## 4. Обучение YOLOv8n

In [ ]:
run_dir = RUNS / RUN_NAME
last_pt = run_dir / 'weights' / 'last.pt'
best_pt = run_dir / 'weights' / 'best.pt'

if last_pt.exists():
    print(f'🔄 Resume: {last_pt}')
    model = YOLO(str(last_pt))
    extra = {'resume': True}
elif best_pt.exists():
    print(f'🔄 Fine-tune: {best_pt}')
    model = YOLO(str(best_pt))
    extra = {}
else:
    print('🆕 Новое обучение с COCO weights')
    model = YOLO('yolov8n.pt')
    extra = {}

results = model.train(
    data=str(YAML_PATH),
    epochs=80,
    imgsz=IMG_SIZE,
    batch=16,
    patience=15,
    device=DEVICE,
    workers=2,
    project=str(RUNS),
    name=RUN_NAME,
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.001,
    mosaic=1.0,
    mixup=0.1,
    save=True,
    save_period=5,
    plots=True,
    verbose=True,
    **extra,
)
print('✅ Обучение завершено')

---
## 5. Графики обучения

In [ ]:
run_dir = RUNS / RUN_NAME
for plot in ['results.png','confusion_matrix.png','PR_curve.png','F1_curve.png']:
    p = run_dir / plot
    if p.exists():
        print(f'📊 {plot}:')
        display(IPImage(filename=str(p), width=800))

---
## 6. Оценка

In [ ]:
best_pt = RUNS / RUN_NAME / 'weights' / 'best.pt'
if not best_pt.exists():
    print('⚠️ best.pt не найден'); 
else:
    model = YOLO(str(best_pt))
    metrics = model.val(data=str(YAML_PATH), imgsz=IMG_SIZE, verbose=False)

    # FPS
    dummy = np.random.randint(0,255,(IMG_SIZE,IMG_SIZE,3),dtype=np.uint8)
    for _ in range(10): model.predict(dummy, verbose=False)
    ts = []
    for _ in range(100):
        t0=time.perf_counter(); model.predict(dummy,verbose=False); ts.append(time.perf_counter()-t0)
    fps = 1/np.mean(ts)

    params = sum(p.numel() for p in model.model.parameters())

    print(f'\n{"="*50}')
    print(f'📊 YOLOv8n — Результаты')
    print(f'{"="*50}')
    print(f'   mAP50:     {metrics.box.map50:.4f}')
    print(f'   mAP50-95:  {metrics.box.map:.4f}')
    print(f'   Precision:  {metrics.box.mp:.4f}')
    print(f'   Recall:     {metrics.box.mr:.4f}')
    print(f'   FPS:        {fps:.1f}')
    print(f'   Params:     {params/1e6:.1f}M')

    result = {'Model':f'YOLOv8{model_size}','mAP50':f'{metrics.box.map50:.4f}',
              'mAP50-95':f'{metrics.box.map:.4f}','Precision':f'{metrics.box.mp:.4f}',
              'Recall':f'{metrics.box.mr:.4f}','FPS':f'{fps:.1f}','Params(M)':f'{params/1e6:.1f}'}
    pd.DataFrame([result]).to_csv(VIZ/f'metrics_{RUN_NAME}.csv', index=False)
    print(f'\n💾 Метрики: {VIZ/f"metrics_{RUN_NAME}.csv"}')

# Per-class метрики
    for i, name in enumerate(CLASS_NAMES):
        if i < len(metrics.box.ap50):
            print(f'   {name:12s}: AP50={metrics.box.ap50[i]:.4f}')

---
## 7. Визуальный инференс

In [ ]:
best_pt = RUNS / RUN_NAME / 'weights' / 'best.pt'
if best_pt.exists():
    model = YOLO(str(best_pt))
    test_dir = ds_root / 'test' / 'images'
    if not test_dir.exists(): test_dir = ds_root / 'valid' / 'images'
    test_imgs = [p for p in test_dir.iterdir() if p.suffix.lower() in {'.jpg','.png'}]
    samples = random.sample(test_imgs, min(8, len(test_imgs)))

    fig,axes = plt.subplots(2,4,figsize=(20,10)); axes=axes.flatten()
    for ax,p in zip(axes,samples):
        res = model.predict(str(p), imgsz=IMG_SIZE, conf=0.25, verbose=False)
        ann = cv2.cvtColor(res[0].plot(), cv2.COLOR_BGR2RGB)
        ax.imshow(ann); ax.set_title(f'{len(res[0].boxes)} obj',fontsize=10); ax.axis('off')
    for j in range(len(samples),len(axes)): axes[j].axis('off')
    plt.suptitle('YOLOv8n — детекция',fontsize=15,fontweight='bold')
    plt.tight_layout(); plt.savefig(VIZ/'detection_results.png',dpi=150,bbox_inches='tight'); plt.show()

---
## 8. Сводка

In [ ]:
print('\n📥 Артефакты:')
for p,d in [(RUNS/RUN_NAME/'weights/best.pt','Best weights'),
            (RUNS/RUN_NAME/'weights/last.pt','Last checkpoint'),
            (RUNS/RUN_NAME/'results.png','Training curves'),
            (RUNS/RUN_NAME/'confusion_matrix.png','Confusion matrix'),
            (VIZ/'class_distribution.png','Class distribution'),
            (VIZ/'detection_results.png','Detection results')]:
    p=Path(p)
    if p.exists(): print(f'  ✅ {d:25s} — {p.name} ({p.stat().st_size/1e6:.1f} MB)')
    else: print(f'  ⬜ {d}')
print(f'\n   💡 Для resume: Save Version → New Dataset из Output → Add Data при новом запуске')